In [21]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()


  Activating project at `/mnt/d/Aho/Vibe_Project/cDFT/cDFT.jl`


In [22]:
Pkg.add("Plots")
using Clapeyron, cDFT, Plots

   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `/mnt/d/Aho/Vibe_Project/cDFT/cDFT.jl/Project.toml`
    Manifest No packages added to or removed from `/mnt/d/Aho/Vibe_Project/cDFT/cDFT.jl/Manifest.toml`


## General set-up
The cDFT and Clapeyron packages are very closely related. Clapeyron provides all the bulk information while cDFT handles all inhomogeneous calculations. The first step in any cDFT calculation is to first define the model species:

In [23]:
model = PCSAFT(["ethane"])

PCSAFT{BasicIdeal, Float64} with 1 component:
 "ethane"
Contains parameters: Mw, segment, sigma, epsilon, epsilon_assoc, bondvol

From here, we want to define our system conditions. Here, we ethane within a graphite slit at 298.15 K and 10 MPa:

In [24]:
T = 298.15
p = 1e7
v = volume(model, p, T);

With these conditions, we can now define a system structure. We now need to define our interface. We will simply use the Steele potential to represent a graphite surface.

In [25]:
surface = Steele(["graphite"]);

We can now construct our structure:

In [26]:
ρ = [1.]/v

L = cDFT.length_scale(model) # Useful tool to obtain a characteristic length scale for the system, which can be used to non-dimensionalize the problem and choose an appropriate grid size.

width = 5L
bounds = [0.6L,width-0.6L]

structure = cDFT.ExternalField1DCart((p, T), ρ, bounds, (201,), surface, width);

With this, we can now fully define our system:

In [27]:
system = DFTSystem(model, structure)

DFTSystem with 1 component:
 model: PCSAFT{BasicIdeal, Float64}("ethane")
 structure: ExternalField1DCart
 device: CPU

And initialize the profiles:

In [28]:
ρ = cDFT.initialize_profiles(system);

MethodError: MethodError: no method matching initialize_profiles(::PCSAFT{BasicIdeal, Float64}, ::ExternalField1DCart, ::cDFT.PCSAFTSpecies, ::CPU)
The function `initialize_profiles` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  initialize_profiles(::EoSModel, !Matched::Uniform1DCart, ::Any, ::Any)
   @ cDFT /mnt/d/Aho/Vibe_Project/cDFT/cDFT.jl/src/structure/structure.jl:50
  initialize_profiles(::EoSModel, !Matched::TwoPhase3DLamCart, ::Any, ::Any)
   @ cDFT /mnt/d/Aho/Vibe_Project/cDFT/cDFT.jl/src/structure/two_phase.jl:66
  initialize_profiles(::EoSModel, !Matched::cDFT.Uniform3DCart, ::Any, ::Any)
   @ cDFT /mnt/d/Aho/Vibe_Project/cDFT/cDFT.jl/src/structure/structure.jl:80
  ...


I've created some convenience functions to quickly visualize the density profiles:

In [29]:
plot(system, ρ)

print device already activated
print device already activated


Note that this profile is just an initial guess and does not correspond to the true equilibrium profile. This profile must be solved for iteratively such that the following equation is satisfied:
$$ \rho(\mathbf{r}) = \rho^b \exp\left[-\beta\left(\frac{\delta F_\mathrm{res}}{\delta \rho(\mathbf{r})}+V_\mathrm{ext}(\mathbf{r})\right)\right]$$

This is achieved using the `converge!(system, ρ)` function:

In [30]:
converge!(system, ρ);

DimensionMismatch: DimensionMismatch: new dimensions (201, 1) must be consistent with array length 1

We can see the visual difference in the converged profile:

In [31]:
plot(system, ρ)

print device already activated
print device already activated
print device already activated
